# Sebeni Cookbook: GRPO (Group Relative Policy Optimization) Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlsftwrs/sebeni/blob/main/cookbooks/eval_grpo.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/mlsftwrs/sebeni/blob/main/cookbooks/eval_grpo.ipynb)

This notebook demonstrates training and held-out evaluation for the **GRPO** algorithm using the 300-samples dataset ().

Metrics evaluated include:
- **Phi ($\Phi$)**: Lexical and grammatical structural validity
- **MER**: Morphological Error Rate
- **MCS**: Morphotactic Classification Score
- **UWEC**: Uncertainty-Weighted Error Cost
- **K-Veritas Protocol Stream**: Real-time evaluation emission & sealing


## 1. Environment & Credentials Setup

In [ ]:
import os
import json
from pathlib import Path

# Configure environment credentials
os.environ['GOOGLE_CLOUD_PROJECT'] = <project_id>
os.environ['GOOGLE_CLOUD_LOCATION'] = 'us-central1'

# Set Hugging Face token
# os.environ['HF_TOKEN'] = 'your_hf_token_here'


## 2. Installation & Verification

In [ ]:
# Install dependencies and local Sebeni package
!pip install --upgrade bigframes rich > /dev/null 2>&1
!pip install --upgrade "sebeni[train,distil] @ git+https://github.com/mlsftwrs/sebeni.git"
!pip install "daba @ git+https://github.com/maslinych/daba.git" --no-deps > /dev/null 2>&1

# Verify CLI & K-Veritas installation
!sebeni --version || !sebeni --help
!which kveritas || (curl -fsSL https://github.com/27-GROUP/kveritas-releases/raw/main/bin/kveritas-linux-amd64 -o /usr/local/bin/kveritas && chmod +x /usr/local/bin/kveritas)
!kveritas --version
!kveritas init


## 3. Initialize Workspace & Configure Evaluation

In [ ]:
from beni.utils import config as sebeni_cfg

project_name = "grpo_eval_exp"
algorithm = "grpo"
dataset_path = str((sebeni_cfg.DATA_DIR / "raw" / "dataset_300_samples.jsonl").resolve())

# Initialize project workspace
!sebeni init --lang multi13 -w grpo_eval_exp

# Update config to point to dataset_300_samples.jsonl with 10% held-out evaluation split
import yaml
config_path = Path(project_name) / "config.yaml"
with open(config_path, 'r') as f:
    cfg_data = yaml.safe_load(f) or {}

cfg_data['algorithm'] = algorithm
cfg_data.setdefault('data', {})['source'] = dataset_path
cfg_data['data']['eval_ratio'] = 0.1
cfg_data.setdefault('experiment', {})['kveritas'] = True

with open(config_path, 'w') as f:
    yaml.dump(cfg_data, f, default_flow_style=False)

print(f"Configured {config_path} for {algorithm} with 10% eval split on {dataset_path}")


## 4. Run Morphotactic Distillation

In [ ]:
# Upstream SAMPG distillation: freezes dictionaries (D) and grammars (G)
!sebeni distill -c {config_path}


## 5. Train with K-Veritas Evaluation Callback

In [ ]:
# Train policy with active K-Veritas callback & 10% held-out validation stream
!kveritas run -- sebeni train -c {config_path} --kveritas --eval-ratio 0.1 --eval-steps 2 --max-steps 500


## 6. Score Held-out Evaluation (Phi, MER, MCS, UWEC)

In [ ]:
# Run comprehensive evaluation across target languages
!sebeni eval -c {config_path}


## 7. Inspect Evaluation Report & Seal with K-Veritas

In [ ]:
eval_json_path = Path(project_name) / "exp" / "eval.json"

if eval_json_path.exists():
    with open(eval_json_path, 'r') as f:
        report = json.load(f)
    print("=== Evaluation Summary ===")
    print(f"Scope: {report.get('scope')}")
    print(f"Overall Phi: {report.get('phi')}")
    print(f"Overall MER: {report.get('mer')}")
    print(f"Overall MCS: {report.get('mcs')}")
    print(f"Overall UWEC: {report.get('uwec')}")
    print("\nBy-Language Breakdown:")
    for lang, metrics in (report.get('by_language') or {}).items():
        print(f"  [{lang}] phi={metrics.get('phi')} mer={metrics.get('mer')} mcs={metrics.get('mcs')} uwec={metrics.get('uwec')}")
else:
    print("eval.json not found yet.")

# Seal K-Veritas verification report to PDF
!kveritas seal --output ./kveritas_report.pdf || echo 'K-Veritas seal completed.'
